# MedVision-V0 inference with locally prepared KiTS23 data

Prepare and validate scans on your Mac/Linux CPU first using `scripts/prepare_local_data.py`, then upload the **portable-v2 tar archive and JSON sidecar** to `MyDrive/MedMeasure/cache/`. See [local-to-Colab instructions](https://github.com/shaunakkarnik/MedMeasure/blob/main/docs/local-to-colab.md).

This notebook **requires the prepared Drive bundle**. It does not download or preprocess scans and does not fall back to that path if the bundle is absent. It restores data onto runtime-local disk, installs pinned source and rebuilds only the 3,179-row test metadata cache with network downloads blocked. HF Arrow caches from your local machine are excluded so absolute filenames are regenerated for `/content`.

Inference remains the unchanged MedVision-V0 sizing task: official v1.4.0 full test set and its 43-case/825-row diagnostic. Disclose possible checkpoint training exposure; detection is not implemented. Select an A100 GPU and High-RAM if available. Copy this notebook to Drive and run cells individually. Runtime transfer/extraction, environment setup and model loading still use time on the GPU allocation.

Commit/push the new pipeline first and set `CODE_COMMIT` to that full SHA. Use a new `RUN_NAME` for this route; keep it and the code/shard configuration fixed when resuming. This pipeline has targeted CPU checks but real local staging, Drive restoration and GPU inference must still be verified.


## Stream command output
Run this helper before setup. It forwards stdout and stderr into Colab cell output, including progress updates without newlines, and enables unbuffered Python logging in child processes. A nonzero exit still fails the cell. It applies only to commands started after this helper is installed; it cannot attach to an already-running download.


In [ ]:
# Forward child-process output through Python so Colab renders it in this cell.
import codecs
import os
import signal
import subprocess
import sys


def stream_command(command, *, cwd=None, env=None):
    command = list(map(str, command))
    child_env = dict(os.environ if env is None else env)
    child_env.update(PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    with subprocess.Popen(command, cwd=cwd, env=child_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          start_new_session=True) as process:
        try:
            # Read available bytes rather than lines, including progress bars and
            # messages that have no trailing newline yet.
            while block := os.read(process.stdout.fileno(), 4096):
                sys.stdout.write(decoder.decode(block))
                sys.stdout.flush()
            sys.stdout.write(decoder.decode(b'', final=True))
            sys.stdout.flush()
            returncode = process.wait()
        except BaseException:
            # An interrupted cell must not leave its evaluator children running.
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


In [ ]:
REPO_URL = "https://github.com/shaunakkarnik/MedMeasure.git"
CODE_COMMIT = ""  # Full 40-character SHA after committing/pushing this notebook.
CACHE_CODE_COMMIT = ""  # Optional full SHA for the backup helper; defaults to CODE_COMMIT.
RUN_NAME = "kits23-medvision-v0-local-data-v140"
SHARD_SIZE = 100  # Keep fixed within a run; change RUN_NAME if changing this.
MAX_CHUNKS = 1  # First run: one chunk. Set 0 for all remaining chunks.
GIT_NAME = ""  # Only needed for optional results export.
GIT_EMAIL = ""

import os, sys, json, re, shutil, subprocess, tempfile
from pathlib import Path
from google.colab import drive, userdata
from IPython.display import display, Image

if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Set CODE_COMMIT to the full GitHub commit SHA containing this notebook and scripts")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", RUN_NAME):
    raise ValueError("Use a simple RUN_NAME with letters, numbers, dots, underscores or hyphens")
if SHARD_SIZE < 1 or MAX_CHUNKS < 0:
    raise ValueError("SHARD_SIZE must be positive; MAX_CHUNKS must be nonnegative")

def secret(name):
    try:
        return userdata.get(name)
    except userdata.SecretNotFoundError:
        return None

# Tokens stay out of Git remotes, command arguments and saved files.
def git(*args, cwd=None, authenticated=False):
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    with tempfile.TemporaryDirectory() as directory:
        if authenticated:
            token = secret('GITHUB_TOKEN')
            if not token:
                raise ValueError('Add GITHUB_TOKEN to Colab Secrets and enable notebook access')
            askpass = Path(directory)/'askpass'
            askpass.write_text('#!/usr/bin/env python3\nimport os, sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["MEDMEASURE_GIT_TOKEN"])\n')
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), MEDMEASURE_GIT_TOKEN=token)
        return subprocess.check_output(['git', '-c', 'credential.helper=', *map(str,args)], cwd=cwd, env=env, text=True).strip()

REPO = Path('/content/MedMeasure')
if not REPO.exists():
    git('clone', REPO_URL, REPO, authenticated=bool(secret('GITHUB_TOKEN')))
if git('status', '--porcelain', cwd=REPO):
    raise ValueError('Runtime checkout has local changes; preserve them before resetting it')
git('fetch', 'origin', CODE_COMMIT, cwd=REPO, authenticated=bool(secret('GITHUB_TOKEN')))
git('checkout', '--detach', CODE_COMMIT, cwd=REPO)
assert git('rev-parse', 'HEAD', cwd=REPO) == CODE_COMMIT
lock = json.loads((REPO/'configs/kits23-pilot.json').read_text())
print('Using MedMeasure commit:', CODE_COMMIT)

# Fetch only the cache helper; keep the inference checkout at CODE_COMMIT.
cache_commit = CACHE_CODE_COMMIT or CODE_COMMIT
if not re.fullmatch(r"[0-9a-f]{40}", cache_commit):
    raise ValueError('CACHE_CODE_COMMIT must be blank or a full commit SHA')
git('fetch', 'origin', cache_commit, cwd=REPO, authenticated=bool(secret('GITHUB_TOKEN')))
CACHE_HELPER = Path('/content/cache_colab_data.py')
CACHE_HELPER.write_text(git('show', cache_commit+':scripts/cache_colab_data.py', cwd=REPO)+'\n')


## Mount Drive and require the portable bundle
Check that both files exist before continuing to the large GPU environment installation.


In [ ]:
drive.mount('/content/drive')
PERSIST = Path('/content/drive/MyDrive/MedMeasure')/RUN_NAME
PERSIST.mkdir(parents=True, exist_ok=True)
AUDIT = PERSIST/'audit'
SHARDS = PERSIST/'shards'
OUTPUT = PERSIST/'evaluation'
DATA = Path('/content/medmeasure-data')
MODEL = Path('/content/MedVision-V0-7B')
UPSTREAM = Path('/content/MedVision')
VENV = Path('/content/medmeasure-venv')
PYTHON = VENV/'bin/python'
DATA_ARCHIVE = Path('/content/drive/MyDrive/MedMeasure/cache')/f"kits23-{lock['dataset_revision']}-portable-v2.tar"
print('Persistent output:', PERSIST)
stream_command(['nvidia-smi'])
print('Runtime disk free (GiB):', round(shutil.disk_usage('/content').free/2**30, 1))

if not DATA_ARCHIVE.is_file() or not DATA_ARCHIVE.with_name(DATA_ARCHIVE.name+'.json').is_file():
    raise FileNotFoundError('Upload the completed portable-v2 archive AND JSON sidecar to MyDrive/MedMeasure/cache before continuing')
cache_info = json.loads(DATA_ARCHIVE.with_name(DATA_ARCHIVE.name+'.json').read_text())
if cache_info.get('format') != 2 or cache_info.get('dataset_revision') != lock['dataset_revision']:
    raise ValueError('This notebook requires a matching portable format-2 dataset bundle')


## Install the pinned evaluation environment
An isolated Python 3.11 environment avoids replacing Colab's notebook packages. Installation uses the pinned upstream launcher's frozen requirements with `--no-deps`, then installs benchmark and vendored evaluator source without resolving their conflicting package metadata pins. This is a sizable installation. Installation errors should be investigated before staging or inference; do not silently upgrade the evaluator.


In [ ]:
if not UPSTREAM.exists():
    git('clone', 'https://github.com/YongchengYAO/MedVision.git', UPSTREAM)
git('checkout', '--detach', lock['benchmark_commit'], cwd=UPSTREAM)
if git('status', '--porcelain', cwd=UPSTREAM):
    raise ValueError('Upstream checkout must be unchanged')
stream_command([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'])
if not PYTHON.exists():
    stream_command([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(VENV)])
ENV = os.environ.copy()
ENV['PATH'] = str(VENV/'bin') + os.pathsep + ENV['PATH']
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', MPLBACKEND='Agg',
           PYTHONUNBUFFERED='1', PYTHONIOENCODING='utf-8')
if secret('HF_TOKEN'):
    ENV['HF_TOKEN'] = secret('HF_TOKEN')

def run(*args):
    stream_command(args, cwd=REPO, env=ENV)

def script(name, *args):
    run(PYTHON, REPO/'scripts'/name, *args)

# Install torch first so source-package build steps can import it.
run(PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1')
run(PYTHON, '-m', 'pip', 'install', '--no-deps', '-r', UPSTREAM/'requirements/requirements_eval_medvision-v0.txt')
# Build from a copy to keep the inspected upstream Git checkout clean.
with tempfile.TemporaryDirectory() as directory:
    source = Path(directory)/'MedVision'
    shutil.copytree(UPSTREAM, source, ignore=shutil.ignore_patterns('.git'))
    run(PYTHON, '-m', 'pip', 'install', '--no-deps', source)
    run(PYTHON, '-m', 'pip', 'install', '--no-deps', source/'src/medvision_bm/medvision_lmms_eval')
run(PYTHON, '-c', 'import torch, vllm, datasets, transformers; assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0)); print("VRAM GiB:", torch.cuda.get_device_properties(0).total_memory/2**30); print("vLLM:", vllm.__version__); assert datasets.__version__ == "3.6.0"; assert transformers.__version__ == "4.54.1"')


## Restore processed scans and rebuild destination metadata

The archive is checksummed before extraction and existing data is never overwritten. On a fresh VM this takes time but performs no scan download or reorientation. The restored `src` package is installed with `--no-deps` so the pinned GPU dependencies stay intact. Only the test metadata cache is regenerated, using the pinned local loader, and all row identities, measurements, spacing and destination file paths are checked. Unexpected download attempts fail.

If restoration is interrupted, preserve/remove the partial runtime data directory explicitly before retrying. Keep the complete Drive archive and sidecar.


In [ ]:
if not DATA.exists() or not any(DATA.iterdir()):
    run(PYTHON, CACHE_HELPER, 'restore', '--data-dir', DATA, '--archive', DATA_ARCHIVE,
        '--config', REPO/'configs/kits23-pilot.json')
else:
    print('Keeping existing runtime data; offline validation below will check it.')
run(PYTHON, '-m', 'pip', 'install', '--no-deps', DATA/'src')
script('prepare_inference_data.py', '--data-dir', DATA)
print(json.loads((DATA/'inference-data-ready.json').read_text()))


## Stage pinned metadata and prepare the frozen evaluation
This cell checks existing audit/shard manifests before reuse. Keep `SHARD_SIZE` unchanged when resuming.


In [ ]:
if not AUDIT.exists():
    script('prepare_data.py', '--plan', DATA/'metadata/benchmark_plan_biometry_v1.4.0.json.gz',
           '--legacy-plan', DATA/'metadata/benchmark_plan_biometry_v1.0.0.json.gz',
           '--legacy-plan', DATA/'metadata/benchmark_plan_biometry_v1.1.1.json.gz', '--output', AUDIT)
# Regenerate in scratch and compare, so a partially created audit is never reused.
with tempfile.TemporaryDirectory() as directory:
    fresh = Path(directory)/'audit'
    script('prepare_data.py', '--plan', DATA/'metadata/benchmark_plan_biometry_v1.4.0.json.gz',
           '--legacy-plan', DATA/'metadata/benchmark_plan_biometry_v1.0.0.json.gz',
           '--legacy-plan', DATA/'metadata/benchmark_plan_biometry_v1.1.1.json.gz', '--output', fresh)
    for path in fresh.iterdir():
        if not (AUDIT/path.name).exists() or (AUDIT/path.name).read_bytes() != path.read_bytes():
            raise ValueError('Saved audit differs from regenerated pinned metadata; use a new run or repair the incomplete audit')
if not SHARDS.exists():
    script('plan_evaluation.py', '--manifest', AUDIT/'test.jsonl', '--output', SHARDS, '--shard-size', SHARD_SIZE)
if json.loads((SHARDS/'plan.json').read_text())['max_rows_per_shard'] != SHARD_SIZE:
    raise ValueError('Saved shard size changed; restore SHARD_SIZE or use a new RUN_NAME')
print(json.loads((AUDIT/'audit.json').read_text())['groups'])


## Stage the model
This downloads only the pinned model weights. Scans have already been restored and validated.


In [ ]:
script('stage_model.py', '--output', MODEL, '--resume')
print('Runtime disk free (GiB):', round(shutil.disk_usage('/content').free/2**30, 1))


## Geometry check and visual review
Check 20 training/validation-reference rows spanning target sizes before inference. The cell fails on mismatched geometry. Review the displayed overlays (and remaining PNGs in Drive). These native-grid diagnostics do not verify the VLM's rendered images. If interrupted, preserve the old directory and use a fresh geometry directory.


In [ ]:
GEOMETRY = PERSIST/'geometry-smoke'
if not GEOMETRY.exists():
    script('check_geometry.py', '--manifest', AUDIT/'train.jsonl', '--dataset-dir', DATA/'Datasets/KiTS23',
           '--output', GEOMETRY, '--limit', 20)
geometry = json.loads((GEOMETRY/'summary.json').read_text())
if geometry['failed'] or geometry['checked'] != 20:
    raise ValueError('Geometry gate failed or incomplete; inspect geometry results before continuing')
for path in sorted(GEOMETRY.glob('*.png'))[:4]:
    display(Image(filename=str(path)))
print('Review all overlays in:', GEOMETRY)


In [ ]:
GEOMETRY_REVIEWED = False  # Set True after inspecting the overlays.
if not GEOMETRY_REVIEWED:
    raise ValueError('Inspect the geometry overlays, then set GEOMETRY_REVIEWED=True')
COMMON = ['--upstream', UPSTREAM, '--data-dir', DATA, '--model-path', MODEL,
          '--audit', AUDIT, '--shards', SHARDS, '--output', OUTPUT]
script('colab_baseline.py', 'smoke', *COMMON)
print(json.loads((OUTPUT/'chunks/smoke/completed.json').read_text()))


## Run evaluation chunks
The default evaluates one 100-row chunk first. After it succeeds, set `MAX_CHUNKS=0` in the configuration cell (or directly below) and rerun this cell to finish pending chunks. Results stay in Drive. A smoke run and failed attempts are excluded from final scores. A failed generation counts through upstream metrics; a crashed/incomplete chunk must be retried rather than silently dropped.


In [ ]:
script('colab_baseline.py', 'run', *COMMON, '--max-chunks', MAX_CHUNKS)


## Score complete results
This refuses missing, duplicate, unexpected or modified sample logs. Full and diagnostic results come from the same inference run. MAE/MRE use successful finite rows; failure and below-10%-error denominators are explicit. Keep upstream aggregate results too. Summary files are not overwritten automatically.


In [ ]:
script('colab_baseline.py', 'score', *COMMON)
for name in ['test', 'test_legacy_disjoint']:
    print(name, json.loads((OUTPUT/f'{name}-summary.json').read_text())['groups']['all'])


## Optional: prepare results for GitHub
Export compact JSON/JSONL predictions, selected-row manifests, configuration, package versions and GPU information. Scans, model weights, overlays and failed attempts stay in Drive. The export is made on a new `results/<RUN_NAME>` branch based on the latest `origin/main`, separate from the pinned inference checkout. Add `GITHUB_TOKEN` (repository Contents read/write permission), `GIT_NAME` and `GIT_EMAIL` first.

Run the next cell to stage and inspect the export, then the following cell to commit and push it. The notebook editor's **Save a copy in GitHub** only saves the notebook, not these result files.


In [ ]:
if not GIT_NAME or not GIT_EMAIL:
    raise ValueError('Set GIT_NAME and GIT_EMAIL in the configuration cell')
if not all((OUTPUT/f'{name}-summary.json').exists() for name in ['test', 'test_legacy_disjoint']):
    raise ValueError('Score the complete run before exporting')
EXPORT = Path('/content/MedMeasure-results')
BRANCH = 'results/' + RUN_NAME
if EXPORT.exists():
    raise ValueError('Export checkout already exists; inspect it or use its prepared commit/push cell below')
git('clone', REPO_URL, EXPORT, authenticated=True)
git('switch', '-c', BRANCH, 'origin/main', cwd=EXPORT)
DEST = EXPORT/'results'/RUN_NAME
if DEST.exists():
    raise ValueError('This results directory already exists in GitHub; choose a distinct RUN_NAME')
DEST.mkdir(parents=True)
for path in AUDIT.iterdir():
    shutil.copy2(path, DEST/path.name)
shutil.copytree(SHARDS, DEST/'shards')
for name in ['evaluation.json', 'test-summary.json', 'test_legacy_disjoint-summary.json']:
    shutil.copy2(OUTPUT/name, DEST/name)
for chunk in sorted((OUTPUT/'chunks').glob('shard-*')):
    marker = json.loads((chunk/'completed.json').read_text())
    attempt = chunk/marker['attempt']
    target = DEST/'chunks'/chunk.name
    target.mkdir(parents=True)
    shutil.copy2(chunk/'completed.json', target/'completed.json')
    for name in ['manifest.json', 'selected_rows.json', 'preflight.json', 'pip-freeze.txt', 'gpu.txt', 'executed-command.json', 'summary.json']:
        shutil.copy2(attempt/name, target/name)
    shutil.copytree(attempt/'results', target/'results')
(DEST/'README.md').write_text('MedVision-V0 official KiTS23 v1.4.0 axial sizing baseline.\n'
    'See evaluation.json for pinned code, model, data and shard provenance.\n'
    'Main split has potential checkpoint training exposure. The legacy-disjoint diagnostic is not proven unseen across all tasks.\n'
    'Scans, weights, geometry PNGs, smoke inference and incomplete attempts are excluded.\n')
git('config', 'user.name', GIT_NAME, cwd=EXPORT)
git('config', 'user.email', GIT_EMAIL, cwd=EXPORT)
git('add', '--', 'results/'+RUN_NAME, cwd=EXPORT)
print(git('diff', '--cached', '--stat', cwd=EXPORT))
print('Export size (MiB):', round(sum(p.stat().st_size for p in DEST.rglob('*') if p.is_file())/2**20, 2))


In [ ]:
# Run after inspecting the staged export above. This publishes the results branch.
git('commit', '-m', 'Add MedVision-V0 KiTS23 baseline results: '+RUN_NAME, cwd=EXPORT)
git('push', '-u', 'origin', BRANCH, cwd=EXPORT, authenticated=True)
print('Results pushed to branch:', BRANCH)
print('Locally: git fetch origin')
print('Then: git merge origin/' + BRANCH)


If pushing fails after the commit succeeds, retry only `git('push', '-u', 'origin', BRANCH, cwd=EXPORT, authenticated=True)`.

Locally, merge the results branch into your working branch after preserving local changes. Then ask Codex to inspect `results/<RUN_NAME>/`. Alternatively keep results on their branch and check it out locally. You can also download the Drive output without GitHub.

Upstream references: [pinned launcher](https://github.com/YongchengYAO/MedVision/blob/45fa8e52684a65da6e1c0ef6ef422d5d4c3e94bd/script/benchmark-TL/eval__MedVision-V0-7B__TL.sh), [Colab storage and runtime limits](https://research.google.com/colaboratory/faq.html). See `docs/colab-runbook.md` for recovery and limitations.
